In [25]:

import ollama
def ask_llm(inp):
    
    response = ollama.chat(
          model = "llama3:8b",
          messages = [{
              "role":"user",
              "content":inp
          }],
        stream = True
    )
    for res in response:
        print(res["message"]["content"],end="",flush=True)
        

In [26]:
while True:
    inp = input("enter your input:")
    ask_llm(inp)

enter your input: hello


Hello! It's nice to meet you. Is there something I can help you with, or would you like to chat?

KeyboardInterrupt: Interrupted by user

In [27]:
 pip install langchain-community pypdf

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [28]:
import sys
print(sys.executable)
!{sys.executable} -m pip install langchain-community pypdf

C:\Users\ssram\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe
Defaulting to user installation because normal site-packages is not writeable


In [29]:
from langchain_community.document_loaders import PyPDFLoader
FILE_PATH = r"C:\Users\ssram\Downloads\Machine_Learning_Theory_2_Pages.pdf"

loader = PyPDFLoader(FILE_PATH)
docs = loader.load()

print(docs[0].page_content)

Machine Learning (ML) - Theory Overview
Machine Learning is a branch of Artificial Intelligence that enables computers to learn patterns from
data and make decisions or predictions without being explicitly programmed for every task. ML
systems improve their performance as they are exposed to more data. The primary goal is to build
models that can generalize from training data to unseen data.
Types of Machine Learning:
Supervised Learning uses labeled data where the correct output is known. Common tasks include
classification and regression.
Unsupervised Learning works with unlabeled data to discover hidden patterns, clusters, or
relationships.
Reinforcement Learning trains an agent to make decisions through rewards and penalties while
interacting with an environment.


In [30]:
!pip install langchain langchain_community langchainhub chromadb langchain-docling

Defaulting to user installation because normal site-packages is not writeable


In [31]:
!pip install langchain-chroma

Defaulting to user installation because normal site-packages is not writeable


In [32]:
!pip install langchain_huggingface

Defaulting to user installation because normal site-packages is not writeable


In [33]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 330, chunk_overlap = 20)
texts = text_splitter.split_documents(docs)

In [34]:
print(texts[0])
print(texts[1])

page_content='Machine Learning (ML) - Theory Overview
Machine Learning is a branch of Artificial Intelligence that enables computers to learn patterns from
data and make decisions or predictions without being explicitly programmed for every task. ML' metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-06-17T15:14:21+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-06-17T15:14:21+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'C:\\Users\\ssram\\Downloads\\Machine_Learning_Theory_2_Pages.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}
page_content='systems improve their performance as they are exposed to more data. The primary goal is to build
models that can generalize from training data to unseen data.
Types of Machine Learning:
Supervised Learning uses labeled data where the correct output is known. Common tasks include
classification and regression.' metadata

In [35]:
!pip install sentence-transformers

Defaulting to user installation because normal site-packages is not writeable


In [36]:
model = "BAAI/bge-large-en-v1.5"

In [37]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name = "BAAI/bge-large-en-v1.5")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [39]:
from langchain_chroma import Chroma
vector_store = Chroma(
    collection_name = "sample_RAG",
    embedding_function=embeddings,
persist_directory = "./chroma_langchain_db",
)

In [40]:
vector_store._collection.count()

0

In [41]:
from langchain_community.vectorstores.utils import filter_complex_metadata
texts = filter_complex_metadata(texts)

In [42]:
texts[0]

Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-06-17T15:14:21+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-06-17T15:14:21+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'C:\\Users\\ssram\\Downloads\\Machine_Learning_Theory_2_Pages.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='Machine Learning (ML) - Theory Overview\nMachine Learning is a branch of Artificial Intelligence that enables computers to learn patterns from\ndata and make decisions or predictions without being explicitly programmed for every task. ML')

In [43]:
vector_store.add_documents(texts)

['74ace919-b1d3-48dd-b7cb-e85f8f87127e',
 'b84452a4-8733-4bab-9a50-b394115aeb3f',
 '3302c1c4-1ee6-4477-bf43-3d8d6df0f94e',
 'e02e5f73-9e18-4634-af37-ce6898dc4c6d',
 'e9f6a65d-adba-4740-a97b-7b70b2a947db',
 '1d7cbef7-f3fe-4b55-85b0-c7df1f65d0bc']

In [44]:
vector_store._collection.count()

6

In [45]:
vector_store._collection.get(ids = ['b84452a4-8733-4bab-9a50-b394115aeb3f'], include = ["embeddings","documents"])

{'ids': ['b84452a4-8733-4bab-9a50-b394115aeb3f'],
 'embeddings': array([[-0.01095835,  0.0019431 ,  0.00937311, ...,  0.01085915,
         -0.00262698,  0.04724511]], shape=(1, 1024)),
 'documents': ['systems improve their performance as they are exposed to more data. The primary goal is to build\nmodels that can generalize from training data to unseen data.\nTypes of Machine Learning:\nSupervised Learning uses labeled data where the correct output is known. Common tasks include\nclassification and regression.'],
 'uris': None,
 'included': ['embeddings', 'documents'],
 'data': None,
 'metadatas': None}

In [46]:
retreiver = vector_store.as_retriever()

In [47]:
query = "what are my interests?"
retreived_docs = retreiver.invoke(query)
context = "\n\n".join(
    [doc.page_content for doc in retreived_docs]
)
print(context)

prompt = f""" you are a RAG assistant.you need to answer the queries based on the provided context.
if the answer is not present int the provided context then politely return "I dont know".
the input is {query} 
the retreived context is {context} """
ask_llm(prompt)

complex real-world problems and improving decision-making across industries.

Applications of Machine Learning include recommendation systems, image recognition, speech
processing, healthcare diagnostics, fraud detection, autonomous vehicles, and natural language
processing. As data availability continues to grow, Machine Learning plays a critical role in solving

Machine Learning (ML) - Theory Overview
Machine Learning is a branch of Artificial Intelligence that enables computers to learn patterns from
data and make decisions or predictions without being explicitly programmed for every task. ML

Key Concepts and Applications
Important concepts in Machine Learning include datasets, features, labels, training, testing, model
evaluation, overfitting, and underfitting. A dataset is divided into training and testing sets. The
model learns from the training data and is evaluated on testing data to measure its performance.
Based on the provided context, I would say that your interests are li

In [48]:
prompt

' you are a RAG assistant.you need to answer the queries based on the provided context.\nif the answer is not present int the provided context then politely return "I dont know".\nthe input is what are my interests? \nthe retreived context is complex real-world problems and improving decision-making across industries.\n\nApplications of Machine Learning include recommendation systems, image recognition, speech\nprocessing, healthcare diagnostics, fraud detection, autonomous vehicles, and natural language\nprocessing. As data availability continues to grow, Machine Learning plays a critical role in solving\n\nMachine Learning (ML) - Theory Overview\nMachine Learning is a branch of Artificial Intelligence that enables computers to learn patterns from\ndata and make decisions or predictions without being explicitly programmed for every task. ML\n\nKey Concepts and Applications\nImportant concepts in Machine Learning include datasets, features, labels, training, testing, model\nevaluation,

In [ ]:
while True:
    inp = input("Enter input/Query: ")
    retrieved_docs=retreiver.invoke(inp)
    context="\n\n".join(
        [doc.page_content for doc in retrieved_docs]
    )
    prompt = f"""
You are a Intelligent RAG assistant and you need to answer quries based on provided context.
If the answer is not present in the provided context then politely return "I donot know"
The input is {inp}
The retrived context is {context}
"""
    ask_llm(prompt)

Enter input/Query:  what are the applications


Based on the provided context